# 16 - Reading the cold constants

**Purpose.** To explain what notebook `15` established about `g`, `R` and the pedestal at -20 C,
and what a reader should now believe about each. `15` is the notebook that talked to the camera
and made these numbers, and is written for someone *checking* the work. This one is written for
someone *deciding what to do next* - specifically, whether session 06's sky frames can be turned
into electrons, and with which numbers.

**What it is not for.** It measures nothing and writes nothing. Every number is read back from
`results/` - `cold_constants.json`, `cold_rungs.csv`, `cold_bias.csv`, and the predecessors they
are judged against: `ptc_constants.json` from session 02, `linearity_constants.json` from session
05 for the panel and the clip, and `bias_constants.json` from session 01. Where arithmetic
appears below it is done on published numbers, to show what a published number is worth; if any
of it disagreed with `results/`, `results/` would be right and this notebook would be the bug.

**It assumes `00_statistics.ipynb`** for why a plane mean over a quarter of a million pixels
resolves a hundredth of a count, and why a spread across repeats is the yardstick a difference
has to beat. It assumes `06_ptc_read.ipynb` for what a photon transfer curve measures and why the
pair difference is the thing that makes it blind to fixed pattern.

**The headline: every constant moved, and the whole of it is worth 1.6%.**

1. **The bench held still.** The two -10 C arms bracket the cold one and agree to 0.01% and 0.08%
   in `g`. That is nine to forty times tighter than the `g` effect itself, and it is what makes
   everything below readable.
2. **It still reproduces session 02**, five weeks and one panel reconfiguration later, to 0.63%
   at gain 50 and 0.19% at gain 200. The -20 C arm is anchored to something outside its own
   evening.
3. **All eight coefficients resolved.** `g`, `R` and the pedestal, at both gains, each moved by
   more than the two warm arms disagree. **`g` was predicted not to move, and it moved** -
   section 3 is about what that is worth and what it is not.
4. **The wrong setpoint was worth 1.6% on `F_sky`** at worst, against L32's own night-to-night
   variation of about 5%. Session 06 would have survived the substitution. It no longer has to
   make one.

**Those four numbers are recomputed in section 1 rather than trusted from here.** A headline
typed into markdown goes stale the moment the session is reshot; the cell under section 1 asks
the same four questions of the published file and will contradict this list if it ever stops
being true. Where the two disagree, the cell is right.

In [ ]:
import json
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "font.size": 8})

RESULTS = pathlib.Path("..") / "results"
read = lambda n: json.loads((RESULTS / n).read_text())

K7 = read("cold_constants.json")          # notebook 15, this session
K5 = read("linearity_constants.json")     # session 05, the panel and the clip
K2 = read("ptc_constants.json")           # session 02, at -10 C
K1 = read("bias_constants.json")          # session 01

rungs = pd.read_csv(RESULTS / "cold_rungs.csv")
bias = pd.read_csv(RESULTS / "cold_bias.csv")

num = lambda d: {int(k): v for k, v in d.items()}
GAINS = sorted(num(K7["system_gain_cold"]["value"]))
COLD_C, WARM_C = -20.0, -10.0
PLANES = ["R", "G1", "G2", "B"]

G_COLD = num(K7["system_gain_cold"]["value"])
G_WARM = {g: K7["vs_session02"]["value"][str(g)]["g_this"] for g in GAINS}
G_S02 = num(K2["system_gain"]["value"])
R_COLD = num(K7["read_noise_cold"]["value"])
PED_COLD = num(K7["pedestal_cold"]["value"])
COEF = K7["temperature_coefficient"]["value"]
RESOLVED = K7["temperature_coefficient_resolved"]["value"]
WOBBLE = K7["arm_disagreement"]["value"]

print("notebook 15 published %d constants on %s, from %d frames"
      % (len(K7), K7["system_gain_cold"]["measured_on"],
         K7["system_gain_cold"]["source_frames"]))
print("gains: %s    setpoints: %s" % (GAINS, sorted(rungs.setpoint_c.unique())))
print("rung table: %d rows over %d arms and %d planes"
      % (len(rungs), rungs.arm.nunique(), rungs.plane.nunique()))
print("bias table: %d rows, %d frames per block"
      % (len(bias), int(bias.n_frames.iloc[0])))
print()
print("cooler duty at the end of a -20 C block: %s%%  (gate 2's bar was 90%%)"
      % K7["cooler_duty_at_cold"]["value"])

## 1. The four questions, answered from the file

Read this cell and you have the session. Everything below it is the working.

In [ ]:
warm_repeat = {q: WOBBLE[q] for q in WOBBLE}
q1 = {g: WOBBLE["g"][str(g)] / G_WARM[g] * 100 for g in GAINS}
q2 = {g: (G_WARM[g] / G_S02[g] - 1) * 100 for g in GAINS}

print("1. DID THE BENCH HOLD STILL?")
for g in GAINS:
    print(f"   gain {g:3d}: the two -10 C arms differ by {q1[g]:.2f}% in g")
print("   " + ("steady -- the -20 C arm between them is clean"
               if max(q1.values()) < 0.5 else
               "NOT steady.  Read every coefficient below as a description of the bench"))

print("\n2. DOES THIS BENCH STILL REPRODUCE SESSION 02?")
for g in GAINS:
    print(f"   gain {g:3d}: {G_WARM[g]:.4f} here vs {G_S02[g]:.4f} in session 02  "
          f"({q2[g]:+.2f}%)")
print("   " + ("reproduced -- two sittings five weeks apart agree, which is stronger "
               "evidence than either alone"
               if max(abs(v) for v in q2.values()) < 1.0 else
               "NOT reproduced.  Something changed between the sessions, and the -20 C "
               "numbers are not publishable until it is found.  Do not average the two"))

print("\n3. DID THE CONSTANTS MOVE WITH TEMPERATURE?")
for q in COEF:
    for g in GAINS:
        d, res = COEF[q][str(g)], RESOLVED[q][str(g)]
        print(f"   {q:>9} gain {g:3d}: {d:+.4f}  "
              f"({'resolved' if res else 'inside the bench wobble -- not resolved'})")

print("\n4. WHAT DID THE WRONG SETPOINT COST SESSION 06?")
print("   priced in section 5, where it lands: F_sky.")

## 2. Why the arms are bracketed instead of interleaved, and what that buys

Every other multi-arm session in this repo interleaves, because session 04 learned the hard way
what happens when two arms differ in *what* is tested and in *when* they ran: one of them turned
out to have been warming, and nothing in the data could separate the two.

Here interleaving is not available. A TEC needs minutes to cross 10 C, so a frame-by-frame
rotation would spend the session in transit and none of it in band.

**Bracketing is the substitute, and it is a weaker instrument with an honest error bar.** Arms 1
and 3 sit either side of the -20 C leg. Anything that drifted across the evening - the backlight
warming, the room, the camera - lands in the gap between them. So the uncertainty on every
temperature coefficient is **half that gap**, not the scatter inside a single arm, which is
typically ten times smaller and would flatter the result enormously.

**It came out sharper than it was designed to be.** The two warm arms agree to 0.01% in `g` at
gain 50 and 0.08% at gain 200, so the bar a temperature effect had to clear turned out to sit far
below the effects themselves. That is the evening's good behaviour rather than the method's
merit - a bracket is only as good as the hours it spans - but it is why all eight coefficients
resolve instead of one or two, and it is why the weak one below is weak for an interesting reason
rather than a boring one.

**Here is what bracketing still cannot do, and it is not a small thing.** It bounds whatever
drifted *in time*. It says nothing about whatever changes with the *setpoint* while not being the
sensor's temperature - above all the TEC itself, which runs at roughly twice the duty cycle at
-20 C as at -10 C (section 6 has the numbers). Nothing in a bracket separates "the sensor is ten
degrees colder" from "the cooler is working twice as hard", because the two arrive together and
leave together.

So every coefficient below means **"the difference between this rig at -20 C and this rig at
-10 C"**. That is precisely the quantity session 06 needs, because session 06's frames were taken
by this rig with its cooler working exactly that hard. It is not a sensor property, and it is not
a number to carry to another camera.

The plot below shows the pedestals themselves, per plane and per arm, which is the cleanest
place to see the three arms with the naked eye. The table under it is the whole argument: every
coefficient beside the bar it had to clear, weakest margin first.

In [ ]:
fig, ax = plt.subplots(1, len(GAINS), figsize=(4.6 * len(GAINS), 3.2))
ax = np.atleast_1d(ax)

for k, g in enumerate(GAINS):
    d = bias[bias.gain == g]
    for arm, mark in zip(sorted(d.arm.unique()), "os^"):
        s = d[d.arm == arm]
        ax[k].scatter(range(len(s)), s.pedestal, marker=mark,
                      label=f"{arm} ({s.setpoint_c.iloc[0]:.0f} C)")
    ax[k].set_xticks(range(len(PLANES)))
    ax[k].set_xticklabels(PLANES)
    ax[k].set_title(f"gain {g}: pedestal per plane per arm")
    ax[k].set_ylabel("ADC counts")
    ax[k].legend(fontsize=7)
fig.tight_layout()

# Every coefficient against its own bar, at both gains.  margin_x is the whole
# verdict in one number: how many times the bench's own disagreement the effect
# is.  coef_pct is the same effect as a fraction of the -10 C value, which is the
# currency the prose above quotes and the one that travels between quantities.
COLD_VAL = {"g": G_COLD, "R_counts": R_COLD, "pedestal": PED_COLD,
            "R_e": num(K7["read_noise_cold_e"]["value"])}

rows = []
for q in COEF:
    for g in GAINS:
        c, w = COEF[q][str(g)], WOBBLE[q][str(g)]
        rows.append({"quantity": q, "gain": g, "coefficient": c,
                     "coef_pct": 100 * c / (COLD_VAL[q][g] - c),
                     "bench_wobble": w,
                     "margin_x": abs(c) / w if w else np.nan,
                     "resolved": RESOLVED[q][str(g)]})
comp = pd.DataFrame(rows).sort_values("margin_x")
print("every coefficient, and the bar it had to clear -- weakest margin first:")
print(comp.round(5).to_string(index=False))

lo = comp.iloc[0]
print(f"\nweakest: {lo.quantity} at gain {lo.gain:.0f}, only {lo.margin_x:.1f}x the bench's own "
      f"wobble -- resolved by the rule, and barely")
print(f"strongest: {comp.iloc[-1].quantity} at gain {comp.iloc[-1].gain:.0f}, "
      f"{comp.margin_x.max():.0f}x")

## 3. `g` was predicted not to move. It moved.

System gain is set by the sense-node capacitance and the ADC voltage reference. Neither has a
strong temperature coefficient over ten degrees, so the prediction written into
`protocols/07-cold-constants.md` before the data existed was **`g` barely moves**. The plan was
that this would be a good null: if `g` were flat, the substitution session 06 would otherwise
have made was harmless, and *knowing* it was harmless beats hoping so.

**The null did not hold.** `g` fell **0.45% at gain 50 and 0.74% at gain 200** going from -10 C
to -20 C, and the bar it had to clear - the two warm arms' own disagreement - it cleared by
**43x and 9x**. This is not the bench: section 2's table has the bench in the neighbouring
column, an order of magnitude quieter than the effect it is being asked to hide.

The prediction is recorded here because it was wrong. A prediction dropped quietly when it fails
is not a prediction, and the protocol wrote it down in advance precisely so that this paragraph
would have to be written.

**Half a percent over ten degrees is a believable size, which is the reassuring part.** It works
out at 0.045%/C, ordinary drift for an uncompensated voltage reference and nothing to raise an
eyebrow at. A `g` that had moved 10% would mean something was broken; a `g` that moves half a
percent means the electronics are behaving like electronics.

**What this session cannot say is which term moved.** Sense-node capacitance, the reference
voltage, and the ADC's own transfer all sit inside the single number `g`, and one evening at two
setpoints separates none of them. It also cannot separate the sensor's temperature from the
cooler's workload, for the reason section 2 gave: at -20 C the TEC runs at roughly twice the
duty. "Temperature coefficient" is shorthand here for "setpoint coefficient of this whole rig".

**Two points define a line and measure nothing about its shape.** There is no evidence here that
the effect is linear in temperature, and a coefficient quoted per degree - as the one above is,
for intuition - is an average over the interval and not a slope at a point. Nothing in this repo
interpolates it to a third setpoint, and nothing should until a third setpoint has been shot.

**Why the finding changes so little in practice.** `g` multiplies every electron figure, so
0.45% on `g` is 0.45% on every electron figure - and section 5 shows the total damage the wrong
setpoint would have done, `g` and pedestal together, came to 1.6%. The value of having measured
it is not that it was large. It is that the alternative was an unbounded guess, and now it is a
bounded number sitting in `cold_constants.json` with its own uncertainty.

The curves below are the raw evidence: variance against signal, one line per arm, at each gain.
Same rungs, same panel, same ROI - only the temperature differs. Look for three lines that lie
on top of each other and separate only in slope; a line that is *offset* rather than tilted would
be a read-noise or pedestal problem wearing a gain costume.

In [ ]:
fig, ax = plt.subplots(1, len(GAINS), figsize=(4.6 * len(GAINS), 3.4))
ax = np.atleast_1d(ax)

for k, g in enumerate(GAINS):
    d = rungs[(rungs.gain == g) & rungs.usable]
    for arm, mark in zip(sorted(d.arm.unique()), "os^"):
        s = d[d.arm == arm].groupby("rung").agg(
            signal=("signal", "mean"), var_pair=("var_pair", "mean"),
            setpoint_c=("setpoint_c", "first")).sort_values("signal")
        ax[k].plot(s.signal, s.var_pair, mark + "-", ms=3, lw=0.8,
                   label=f"{arm} ({s.setpoint_c.iloc[0]:.0f} C)")
    ax[k].set(xscale="log", yscale="log", xlabel="signal, ADC counts",
              ylabel="pair variance, counts^2", title=f"gain {g}")
    ax[k].legend(fontsize=7)
fig.tight_layout()

print("g per arm and gain, mean over the four CFA planes (e- per ADC count):")
gt = bias[["arm", "gain"]].drop_duplicates()
tbl = pd.DataFrame({"g_cold": pd.Series(G_COLD), "g_warm_mean": pd.Series(G_WARM),
                    "g_session02": pd.Series(G_S02).reindex(GAINS)})
tbl["cold_vs_warm_pct"] = 100 * (tbl.g_cold / tbl.g_warm_mean - 1)
tbl["warm_vs_s02_pct"] = 100 * (tbl.g_warm_mean / tbl.g_session02 - 1)
print(tbl.round(4).to_string())

## 4. Read noise fell as predicted; the pedestal is still the one that matters

Two predictions, and they carry very different consequences. Both came true. One of them came
true for a reason that turns out to be false at gain 50, which is worth more than either.

**`R` fell, and falling was the ordinary expectation** - read noise has a thermal component and
colder is quieter. It came down **2.1% at gain 50 and 1.0% at gain 200** in counts, 2.5% and
1.7% once converted to electrons through the cold `g`.

**The claim that it does not matter needs checking at your gain, not in general.** The model's
read term is `R^2/t`, and the sentence written before the session was that sky shot noise buries
it. The cell below runs that on the measured numbers, and it holds only at gain 200: there read
noise is **2% of the variance in a 30 s sub**, genuinely negligible. At gain 50 it is **33%** -
read noise and sky shot noise are within a factor of two of each other. The reason is `g`: the
same 0.9 counts of read noise is **4.9 e- at gain 50 and 1.0 e- at gain 200**, because gain 50
puts six times more charge behind every count.

A few percent on `R` still moves no decision. But it moves no decision because a few percent of
a third is a fraction of a percent, not because the read term is absent - and at gain 50 with
short subs, the read term is emphatically present. That is a fact about session 06's exposure
choice, not about this session, and it is flagged here because this is where it became visible.

**The pedestal is the one that can ruin a number outright.** `F_sky` is defined on the
pedestal-subtracted frame, so a pedestal error goes into the sky rate count for count with
nothing to damp it. At gain 50 the pedestal is about 65 counts and 120 s of L32's green sky is
only about 35 counts above it - so a one-count pedestal error would be a **3% error in the sky
rate**, and at 30 s it would be over 11%. That arithmetic is the whole reason this session was worth
an evening.

**The measured shift is comfortably under a count**: **+0.16 counts at gain 50 and +0.44 at
gain 200**, and the sign is the interesting part. Colder reads *higher*. That rules out dark
current, which falls with temperature and would have pushed the level down. What it leaves is
the black-level circuit and the harder-working TEC, and this session separates neither.

**One of the two pedestal numbers is barely resolved, and it is the gain 50 one.** Its shift is
**1.1x** the warm arms' own disagreement, against **82x** at gain 200. Read the gain 50 pedestal
coefficient as "about a sixth of a count, and this session can only just see it". Read the
gain 200 one as measured. The rule that published it called both resolved, and the rule is right
by its own terms - but a margin of 1.1x and a margin of 82x are not the same claim, and only the
table says so.

In [ ]:
L32_GREEN_E_PER_S = 1.594        # inherited, not ours yet -- a prediction being priced, not used
SUBS_S = [30.0, 120.0]

rows = []
for g in GAINS:
    dped = COEF["pedestal"][str(g)]
    for t in SUBS_S:
        sky_counts = L32_GREEN_E_PER_S * t / G_COLD[g]
        rows.append({"gain": g, "sub_s": t, "sky_above_pedestal_counts": sky_counts,
                     "pedestal_shift_counts": dped,
                     "F_sky_error_pct": 100 * dped / sky_counts})
price = pd.DataFrame(rows)
print("what the pedestal shift alone would have done to F_sky, had it been ignored:")
print(price.round(4).to_string(index=False))

print("\nread noise, and how little of the sub it accounts for:")
rn = []
for g in GAINS:
    R_e = K7["read_noise_cold_e"]["value"][str(g)]
    for t in SUBS_S:
        sky_e = L32_GREEN_E_PER_S * t
        rn.append({"gain": g, "sub_s": t, "R_e": R_e, "sky_shot_e": np.sqrt(sky_e),
                   "read_share_of_variance": R_e ** 2 / (R_e ** 2 + sky_e)})
print(pd.DataFrame(rn).round(4).to_string(index=False))

## 5. What the wrong setpoint actually cost session 06

The question this whole session exists to answer, stated as arithmetic rather than as a feeling.

Two numbers are compared for each of session 06's four cells: `F_sky` computed the way it would
have been with session 02's -10 C constants, and `F_sky` computed with this session's -20 C ones.
The difference is the error that would have been published, silently, with no way for a later
reader to detect it.

**It is priced on L32's sky rate, which is a prediction and not ours.** The point is the *size*
of the correction, not the sky rate itself - session 06 measures that, and this cell is only
showing what an uncorrected number would have been wrong by.

**The answer is 1.6% at worst, and that is survivable.** L32's own night-to-night variation is
about 5%, so the substitution would not have produced a number anyone could have caught, and it
would not have produced a wrong conclusion either. This session turns "probably fine" into a
measured bound, which is the only difference - and the only difference worth an evening.

**The two errors do not combine the same way at the two gains, which is worth seeing rather
than assuming.** At gain 50 and 30 s the pedestal mistake is worth +1.8% and the `g` mismatch
-0.2%: they partly cancel, and +1.6% is what survives. At gain 200 and 30 s the pedestal is
worth +0.8% and `g` +0.6%, and they **add** to +1.4%. The sign of the `g` term is set by which
side of this session's cold value session 02 happened to land on, not by the temperature
coefficient alone.

**That has a practical edge to it.** Correcting `g` alone at gain 50 and leaving the pedestal
would have left +1.8% - *worse* than correcting neither. Constants of the same session are not
a menu; they are a set, and applying half of one is not half a correction.

In [ ]:
CELLS = [(50, 30.0), (50, 120.0), (200, 30.0), (200, 120.0)]

rows = []
for g, t in CELLS:
    sky_counts_cold = L32_GREEN_E_PER_S * t / G_COLD[g]
    # What the wrong path would have done: measure the same raw level, subtract
    # the -10 C pedestal, and scale by the -10 C g.
    raw = PED_COLD[g] + sky_counts_cold
    ped_warm = PED_COLD[g] - COEF["pedestal"][str(g)]
    wrong_e = (raw - ped_warm) * G_S02[g] / t
    right_e = sky_counts_cold * G_COLD[g] / t
    rows.append({"gain": g, "sub_s": t, "F_sky_right": right_e, "F_sky_wrong": wrong_e,
                 "error_pct": 100 * (wrong_e / right_e - 1)})

cost = pd.DataFrame(rows)
print(cost.round(4).to_string(index=False))
worst = float(cost.error_pct.abs().max())
print(f"\nworst error avoided: {worst:.2f}% on F_sky")
print("  " + ("that is inside L32's own night-to-night variation of about 5%, so the "
              "substitution would have been survivable -- and this session is what makes "
              "'survivable' a measurement instead of a hope"
              if worst < 5 else
              "that is larger than L32's night-to-night variation, so the substitution would "
              "have put a real error into a published constant with nothing to reveal it"))

## 6. What the bench itself did, and the one thing it cannot tell us

Three things worth knowing about the evening, none of them the point of the session and all of
them cheap because the frames were being taken anyway.

**`t_sat` per arm** is three readings at one patch colour, and they fell monotonically across the
evening: the panel **brightened about 1.6% across the three arms**. Two things let that be read as the
panel rather than as temperature. The cold arm sits in the *middle* of the trend rather than off
it, so the trend runs in time and not in setpoint - if temperature were driving it, arm 2 would
be the outlier. And the one route by which temperature could move `t_sat` - a rising pedestal
eating the headroom below the clip - is **150 to 400 times too small**: at gain 50 the pedestal
moved 0.16 counts out of some 4030 counts of room below the clip.

So the backlight drifted, and it drifted more than session 05's own bench did (session 05
published 0.31% over its session). Nothing here depends on that, because gate 3 measured `t_sat`
fresh inside every arm instead of carrying a value across. Which is just as well: **this panel is
now about 1.7x brighter than session 05's**, saturating at 36.5 s where session 05 needed 63.3 s
at the same patch colour. That is the reconfiguration, it is expected, and it is exactly why a
carried-across `t_sat` would have put every rung in the wrong place. Measuring it per arm cost
three probe exposures and bought the whole rung ladder.

**The cooler duty at -20 C** is the number that says whether this rig could *work* at that
setpoint rather than merely reach it. It ended a cold gain block at **58%** against gate 2's bar
of 90%, roughly twice what the two warm arms asked of it. (That comparison comes from notebook
15's arm log, which lives in `data/` - it is the session's diary, not a published constant, and
it is named here rather than quoted as one.) The margin below the bar is the headroom for the
self-heating an hour of readout brings. The doubling is also the confound section 2
named: a coefficient measured between these two setpoints has the cooler's extra work folded
into it, and no arrangement of arms could have unfolded it.

**The offset state** appeared in **five of the six** bias blocks and was excluded from both the
pedestal and `R` before either was taken. The hop moves the black level by 0.28-0.57 counts,
which is the same size as the largest coefficient this session publishes - so admitting one
would not have added noise to a result, it would have invented the result. This session does not study the state; session 04 did. It only has
to survive it, and the counts are published so a reader can see it was handled rather than hoped
away. The worst block kept 12 of its 20 frames; one block had no far frame at all.

In [ ]:
tsat = pd.DataFrame(K7["t_sat_per_arm"]["value"]).T
tsat.index.name = "arm"
tsat.columns = [int(c) for c in tsat.columns]
tsat = tsat.sort_index()
print("t_sat per arm, seconds, at session 05's patch colour:")
print(tsat.round(4).to_string())
spread = 100 * (tsat.max() - tsat.min()) / tsat.mean()
print("spread across arms, %:")
print(spread.round(3).to_string())

# Is the drift in time or in setpoint?  Arm 2 is the cold one and sits in the
# middle of the running order, so a monotonic a1 > a2 > a3 is a clock, not a TEC.
mono = all((tsat[g].iloc[0] > tsat[g].iloc[1] > tsat[g].iloc[2]) for g in GAINS)
print("\nmonotonic across the running order (a1 > a2 > a3)? %s%s"
      % (mono, "  -- the panel brightening in time, with the cold arm merely in the middle"
         if mono else "  -- not a clean time trend; look again before blaming the panel"))

T5 = num(K5["t_sat_per_gain"]["value"])
print("\nagainst session 05, same patch colour:")
for g in GAINS:
    here = tsat[g].mean()
    print(f"   gain {g:3d}: {here:7.3f} s here vs {T5[g]:7.3f} s in session 05  "
          f"({T5[g] / here:.2f}x brighter now)")
print("   session 05's own within-session light drift was %.2f%%"
      % K5["light_drift"]["value"])

# Could a rising pedestal explain the t_sat drift?  t_sat is set by the headroom
# between the pedestal and the clip, so price the pedestal shift in that currency.
print("\ncould the pedestal shift explain the t_sat drift instead?")
clip = num(K5["clip_level"]["value"])
for g in GAINS:
    room = clip[g] - PED_COLD[g]
    via_ped = 100 * abs(COEF["pedestal"][str(g)]) / room
    print(f"   gain {g:3d}: pedestal route {via_ped:.4f}%, observed drift {spread[g]:.3f}%"
          f"  -- {spread[g] / via_ped:.0f}x too small")

print("\noffset state in the bias blocks:")
st = bias.groupby(["arm", "gain"]).agg(
    far_frac=("state_far_frac", "first"), n_near=("n_near", "first"),
    separation=("state_separation", "first"))
print(st.round(4).to_string())
print(f"\nblocks with a frame in a far state: "
      f"{int((st.far_frac > 0).sum())} of {len(st)};  "
      f"fewest frames kept: {int(st.n_near.min())} of {int(bias.n_frames.iloc[0])}")
print("  the hop is %.2f-%.2f counts, against a largest coefficient of %.2f counts"
      % (st.separation.min(), st.separation.max(),
         max(abs(COEF["pedestal"][str(g)]) for g in GAINS)))

## 7. What is settled, and what session 06 may now do

The decision this notebook exists to support, spelled out.

In [ ]:
steady = max(WOBBLE["g"][str(g)] / G_WARM[g] for g in GAINS) < 0.005
repeats = max(abs(G_WARM[g] / G_S02[g] - 1) for g in GAINS) < 0.01

print("SETTLED")
print(f"  g at -20 C, gains {GAINS}: " + ", ".join(f"{G_COLD[g]:.4f}" for g in GAINS)
      + " e-/ADC count")
print(f"  R at -20 C, ADC counts:    " + ", ".join(f"{R_COLD[g]:.4f}" for g in GAINS))
print(f"  pedestal at -20 C, counts: " + ", ".join(f"{PED_COLD[g]:.4f}" for g in GAINS))
print()
print("SESSION 06 MAY NOW " + ("PROCEED" if steady and repeats else "NOT PROCEED"))
if steady and repeats:
    print("  The bench held still across the evening and still reproduces session 02, so the")
    print("  -20 C constants are anchored.  17_sky_pair.ipynb reads cold_constants.json for")
    print("  the pedestal and g, and no substitution is being made.")
else:
    print("  " + ("the two warm arms disagree" if not steady else "")
          + ("; " if not steady and not repeats else "")
          + ("this session does not reproduce session 02" if not repeats else ""))
    print("  Reshoot before publishing any electron figure from session 06.")
print()
print("NOT SETTLED, AND DELIBERATELY OUT OF SCOPE")
print("  The shape of the coefficient between -20 and -10 C: two points, so a straight line is")
print("  an assumption and not a measurement.  Nothing in this repo interpolates it, and")
print("  nothing should until a third setpoint exists.")
print("  Which term inside g moved.  Sense node, reference and ADC transfer all sit in the one")
print("  number, and two setpoints on one evening separate none of them.")
print("  Whether the coefficient is the sensor's or the rig's.  At -20 C the TEC runs at about")
print("  twice the duty, and no arrangement of arms separates the two.  It is the right number")
print("  for session 06, whose frames were taken the same way, and the wrong one to carry to")
print("  another camera.")
print("  Whether -20 C is a better setpoint than -10 C.  MISSION fixes -10 C and this session")
print("  characterises an accident; it does not argue for adopting one.")